In [1]:
import numpy as np
import random
import copy

# -----------------------------
# Grid Environment
# -----------------------------
class GridWorld:
    def __init__(self, size=5):
        self.size = size
        self.start = (0, 0)
        self.goal = (size - 1, size - 1)

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        x, y = self.state

        if action == 0:   # up
            x = max(0, x - 1)
        elif action == 1: # down
            x = min(self.size - 1, x + 1)
        elif action == 2: # left
            y = max(0, y - 1)
        elif action == 3: # right
            y = min(self.size - 1, y + 1)

        self.state = (x, y)

        if self.state == self.goal:
            return self.state, 10, True
        else:
            return self.state, -1, False

# -----------------------------
# Q-Learning Agent
# -----------------------------
class Agent:
    def __init__(self, grid_size, alpha=0.1, gamma=0.9, epsilon=0.2):
        self.q_table = np.zeros((grid_size, grid_size, 4))
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def choose_action(self, state):
        if random.uniform(0, 1) < self.epsilon:
            return random.randint(0, 3)
        else:
            x, y = state
            return np.argmax(self.q_table[x, y])

    def update(self, state, action, reward, next_state):
        x, y = state
        nx, ny = next_state

        best_next = np.max(self.q_table[nx, ny])

        self.q_table[x, y, action] += self.alpha * (
            reward + self.gamma * best_next - self.q_table[x, y, action]
        )

# -----------------------------
# Train Agent Locally
# -----------------------------
def train_agent(env, agent, episodes=50):
    for _ in range(episodes):
        state = env.reset()
        done = False

        while not done:
            action = agent.choose_action(state)
            next_state, reward, done = env.step(action)
            agent.update(state, action, reward, next_state)
            state = next_state

    return agent.q_table

# -----------------------------
# Server Aggregation (FedAvg)
# -----------------------------
def aggregate_q_tables(q_tables):
    return np.mean(q_tables, axis=0)

# -----------------------------
# Federated Reinforcement Learning
# -----------------------------
def federated_rl(num_agents=3, rounds=10):
    grid_size = 5
    agents = [Agent(grid_size) for _ in range(num_agents)]
    global_q = np.zeros((grid_size, grid_size, 4))

    for r in range(rounds):
        print(f"\nRound {r+1}")
        local_q_tables = []

        for i, agent in enumerate(agents):
            env = GridWorld(grid_size)

            # Load global Q into agent
            agent.q_table = copy.deepcopy(global_q)

            # Local training
            local_q = train_agent(env, agent, episodes=50)
            local_q_tables.append(local_q)

        # Server aggregation
        global_q = aggregate_q_tables(local_q_tables)

        # Evaluate performance (optional)
        steps = evaluate_policy(global_q, grid_size)
        print(f"Steps to reach goal (global policy): {steps}")

    return global_q

# -----------------------------
# Evaluate Policy
# -----------------------------
def evaluate_policy(q_table, grid_size):
    env = GridWorld(grid_size)
    state = env.reset()
    steps = 0

    while True:
        x, y = state
        action = np.argmax(q_table[x, y])
        state, _, done = env.step(action)
        steps += 1

        if done or steps > 100:
            break

    return steps

# -----------------------------
# Run
# -----------------------------
if __name__ == "__main__":
    final_q = federated_rl(num_agents=3, rounds=10)
    print("\nTraining Completed")


Round 1
Steps to reach goal (global policy): 8

Round 2
Steps to reach goal (global policy): 8

Round 3
Steps to reach goal (global policy): 8

Round 4
Steps to reach goal (global policy): 8

Round 5
Steps to reach goal (global policy): 8

Round 6
Steps to reach goal (global policy): 8

Round 7
Steps to reach goal (global policy): 8

Round 8
Steps to reach goal (global policy): 8

Round 9
Steps to reach goal (global policy): 8

Round 10
Steps to reach goal (global policy): 8

Training Completed
